# 1.使用FUNCTION_CALL模式

In [ ]:
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts.chat import ChatPromptTemplate
from langchain_core.tools.structured import StructuredTool
from langchain_openai import ChatOpenAI
from langchain_classic.agents import initialize_agent, AgentType, create_tool_calling_agent, AgentExecutor
from langchain_community.tools import Tool
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

#读取配置文件里的信息
dotenv.load_dotenv()
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

#获取Tavily搜索工具的实例
search = TavilySearchResults(max_results=3)

search_tool = Tool(
    func=search.run,
    name="Search",
    description="用于检索互联网上的信息",
)

#获取大语言模型
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
chat_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

#提供提示词模板（以ChatPromptTEmplate为例）
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个乐于助人的ai助手，根据用户的提问，必要时调用Search工具，使用互联网检索数据"),
    ("human", "{input}"),
    ("system","{chat_history}"),
    ("placeholder", "{agent_scratchpad}"),
])

#获取记忆的实例
memory = ConversationBufferMemory(
    return_messages=True,
    memory_key="chat_history",
)

#获取Agent的实例，create_tool_calling_agent()
agent = create_tool_calling_agent(
    llm=chat_model,
    prompt=prompt_template,
    tools=[search_tool],
)

#获取AgentExecutor的实例
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool],
    verbose=True,
    memory=memory,
)

result = agent_executor.invoke({"input": "今天的湘潭天气怎么样？"})

print(result)

In [ ]:
result = agent_executor.invoke({"input": "上海的呢？"})

print(result)

# 2.使用ReAct模式

 ### 使用现成的模板

In [ ]:
from langchain_classic import hub
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts.chat import ChatPromptTemplate
from langchain_core.tools.structured import StructuredTool
from langchain_openai import ChatOpenAI
from langchain_classic.agents import initialize_agent, AgentType, create_tool_calling_agent, AgentExecutor, \
    create_react_agent
from langchain_community.tools import Tool
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

#读取配置文件里的信息
dotenv.load_dotenv()
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

#获取Tavily搜索工具的实例
search = TavilySearchResults(max_results=3)

search_tool = Tool(
    func=search.run,
    name="Search",
    description="用于检索互联网上的信息",
)

#获取大语言模型
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
chat_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

#提供提示词模板（以ChatPromptTemplate为例）
prompt_template = hub.pull("hwchase17/react-chat")


#获取记忆的实例
memory = ConversationBufferMemory(
    return_messages=True,
    memory_key="chat_history",
)

#获取Agent的实例，create_react_agent()
agent = create_react_agent(
    llm=chat_model,
    prompt=prompt_template,
    tools=[search_tool],
)

#获取AgentExecutor的实例
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool],
    verbose=True,
    memory=memory,
)

result = agent_executor.invoke({"input": "今天的湘潭天气怎么样？"})

print(result)